In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold

In [2]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   ---------- ----------------------------- 2.1/8.3 MB 8.4 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.3 MB 16.2 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 15.6 MB/s  0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -------- ------------------------------- 7.3/36.6 MB 34.9 MB/s eta 0:00:01
   -------------- ------------------------- 13.1/36.6 MB 32.9 MB/s eta 0:00:01
   ------------------- -------------------- 17.8/36.6 MB 28.8 MB/s eta 0:00:01
   ------------------------- -------------- 23.3/36.6 MB 28.4 MB/s eta 0:00:01
   ------------------------------- -------- 29.1/36.6 MB 28.4 MB/s eta 0:00:01
   -------------------------------------- - 34.9/36.6 MB 28.4 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 26.4 MB/s  0:00:01

   ------ -----

In [4]:
PROJECT_DIR = Path(r"C:\Users\Joseph (YO)\JGM\U\Semestre_10\InteligenciaArtificial\Proyecto_Yuca")

REPORTS_DIR = PROJECT_DIR / "reports"

MANIFEST_PATH = REPORTS_DIR / "images_manifest.csv"

SPLITS_PATH = REPORTS_DIR / "dataset_splits.csv"

# Cargar inventario
df = pd.read_csv(MANIFEST_PATH)

print("Registros cargados:", len(df))
print(df.head())

Registros cargados: 21397
         filename                         filepath             class  \
0  1000015157.jpg  bacterial_blight\1000015157.jpg  bacterial_blight   
1  1001320321.jpg  bacterial_blight\1001320321.jpg  bacterial_blight   
2  1003888281.jpg  bacterial_blight\1003888281.jpg  bacterial_blight   
3  1012426959.jpg  bacterial_blight\1012426959.jpg  bacterial_blight   
4  1018973237.jpg  bacterial_blight\1018973237.jpg  bacterial_blight   

  extension  width  height format mode  size_bytes  \
0      .jpg    800     600   JPEG  RGB      138826   
1      .jpg    800     600   JPEG  RGB       84235   
2      .jpg    800     600   JPEG  RGB      112356   
3      .jpg    800     600   JPEG  RGB      111983   
4      .jpg    800     600   JPEG  RGB      119993   

                                              sha256 status  error  
0  ff91f1393f5fac8c5e11f42a6668a7c0cfe4c7d403385c...  valid    NaN  
1  ea04fae024cc6fff1544796bf26d655617ec7a6fbd6297...  valid    NaN  
2  df7cb6

In [5]:
df_valid = df[df["status"] == "valid"].copy()

df_valid = df_valid.reset_index(drop=True)

print("Imágenes válidas:", len(df_valid))
print("Clases identificadas:", df_valid["class"].nunique())

print(df_valid["class"].value_counts())

Imágenes válidas: 21397
Clases identificadas: 5
class
mosaic_disease          13158
healthy                  2577
green_mottle             2386
brown_streak_disease     2189
bacterial_blight         1087
Name: count, dtype: int64


In [6]:
# Identificar duplicados ya detectados en la auditoría
duplicados = df_valid[
    df_valid["sha256"].duplicated(keep=False)
].sort_values("sha256")

cantidad_duplicados = len(duplicados)

print("Imágenes involucradas en duplicados:", cantidad_duplicados)

if duplicados.empty:
    print("No se encontraron duplicados exactos.")

else:
    display(
        duplicados[
            ["filepath", "class", "sha256"]
        ]
    )

Imágenes involucradas en duplicados: 0
No se encontraron duplicados exactos.


In [7]:
print("Columnas disponibles:")

print(df_valid.columns.tolist())

Columnas disponibles:
['filename', 'filepath', 'class', 'extension', 'width', 'height', 'format', 'mode', 'size_bytes', 'sha256', 'status', 'error']


In [8]:
display(
    df_valid[
        ["filename", "filepath", "class"]
    ].head(20)
)

,filename,filepath,class
0,1000015157.jpg,bacterial_blight\1000015157.jpg,bacterial_blight
1,1001320321.jpg,bacterial_blight\1001320321.jpg,bacterial_blight
2,1003888281.jpg,bacterial_blight\1003888281.jpg,bacterial_blight
3,1012426959.jpg,bacterial_blight\1012426959.jpg,bacterial_blight
4,1018973237.jpg,bacterial_blight\1018973237.jpg,bacterial_blight
5,102039365.jpg,bacterial_blight\102039365.jpg,bacterial_blight
6,1021758544.jpg,bacterial_blight\1021758544.jpg,bacterial_blight
7,1024089865.jpg,bacterial_blight\1024089865.jpg,bacterial_blight
8,102485576.jpg,bacterial_blight\102485576.jpg,bacterial_blight
9,1026467332.jpg,bacterial_blight\1026467332.jpg,bacterial_blight


In [9]:
SEED = 42

df_development, df_test = train_test_split(
    df_valid,
    test_size=0.15,
    random_state=SEED,
    stratify=df_valid["class"]
)

# Asignar el conjunto correspondiente
df_development = df_development.copy()
df_test = df_test.copy()

df_development["split"] = "development"
df_test["split"] = "test"

# Unificar los conjuntos
df_splits = pd.concat(
    [df_development, df_test],
    ignore_index=True
)

print("Desarrollo:", len(df_development))
print("Prueba:", len(df_test))
print("Total:", len(df_splits))

Desarrollo: 18187
Prueba: 3210
Total: 21397


In [10]:
distribution_splits = pd.crosstab(
    df_splits["class"],
    df_splits["split"]
)

distribution_splits

split,development,test
class,,
bacterial_blight,924,163
brown_streak_disease,1861,328
green_mottle,2028,358
healthy,2190,387
mosaic_disease,11184,1974


In [11]:
distribution_percentages = (
    pd.crosstab(
        df_splits["class"],
        df_splits["split"],
        normalize="index"
    ) * 100
)

distribution_percentages.round(2)

split,development,test
class,,
bacterial_blight,85.00,15.00
brown_streak_disease,85.02,14.98
green_mottle,85.00,15.00
healthy,84.98,15.02
mosaic_disease,85.00,15.00


In [12]:
skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

df_development = df_development.copy()

df_development["fold"] = -1

for fold, (train_idx, val_idx) in enumerate(
    skf.split(
        df_development,
        df_development["class"]
    ),
    start=1
):

    df_development.iloc[
        val_idx,
        df_development.columns.get_loc("fold")
    ] = fold

    print(
        f"Fold {fold}: "
        f"Entrenamiento = {len(train_idx)}, "
        f"Validación = {len(val_idx)}"
    )

Fold 1: Entrenamiento = 12124, Validación = 6063
Fold 2: Entrenamiento = 12125, Validación = 6062
Fold 3: Entrenamiento = 12125, Validación = 6062


In [13]:
df_test = df_test.copy()

df_test["fold"] = -1

df_splits = pd.concat(
    [df_development, df_test],
    ignore_index=True
)

# Guardar información de particiones
df_splits.to_csv(
    SPLITS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Particiones guardadas en:", SPLITS_PATH)

Particiones guardadas en: C:\Users\Joseph (YO)\JGM\U\Semestre_10\InteligenciaArtificial\Proyecto_Yuca\reports\dataset_splits.csv


In [14]:
development_paths = set(
    df_splits.loc[
        df_splits["split"] == "development",
        "filepath"
    ]
)

test_paths = set(
    df_splits.loc[
        df_splits["split"] == "test",
        "filepath"
    ]
)

overlap = development_paths.intersection(test_paths)

assert len(overlap) == 0, (
    f"Se encontraron {len(overlap)} imágenes repetidas."
)

print("Verificación correcta: no hay rutas compartidas.")

Verificación correcta: no hay rutas compartidas.


In [15]:
assert set(
    df_development["fold"].unique()
) == {1, 2, 3}

print("Verificación correcta: los tres folds están asignados.")

Verificación correcta: los tres folds están asignados.
